# Transition Matrix Methods for MarkovConsumerType

**Prototype: MC vs TM comparison for a consumption-saving model with discrete Markov states**

This notebook extends Will Du's `Transition_Matrix_Example` to the `MarkovConsumerType`.
The key new element: the agent faces a discrete Markov state $j \in \{0, 1\}$ that
affects the interest rate and unemployment probability.  The transition matrix must
track the joint distribution over $(m, j)$ — market resources and Markov state.

---

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import scipy.sparse.linalg as sp_linalg

from HARK.ConsumptionSaving.ConsMarkovModel import MarkovConsumerType
from HARK.utilities import make_grid_exp_mult, jump_to_grid_1D

# Consistent colors across all MC-vs-TM plots
COLOR_MC = "tab:blue"
COLOR_TM = "tab:orange"

# Named constants
BURNIN = 400
N_MC_BINS = 200

## 1. Model Setup

We define a 2-state Markov model:

- **State 0 (expansion):** higher interest rate, low unemployment
- **State 1 (contraction):** lower interest rate, high unemployment

Both states share the same `PermGroFac = 1.0` so permanent income stays
constant and we can work with a 1D grid over normalized market resources $m$.

**HARK convention:** `MrkvArray` is *row-stochastic* — `MrkvArray[i, j]` = P(go to state $j$ | in state $i$).
The constructor `make_simple_binary_markov` builds it from `Mrkv_p11` (P(stay in 0))
and `Mrkv_p22` (P(stay in 1)).

**Calibration:** Parameters are chosen for pedagogical illustration, not to match
any specific empirical target.  The risk aversion, discount factor, and income process
are loosely based on standard quarterly incomplete-markets defaults.  The two Markov
states are symmetric (`p_stay = 0.9` for both) but have different interest rates and
unemployment probabilities to create visible state-dependent behavior.

In [ ]:
# Pedagogical calibration: symmetric 2-state Markov, quarterly frequency.
# Not based on a specific published calibration — chosen to illustrate TM methods.
p_stay = 0.9  # P(stay in same Markov state)

params = {
    # Preferences
    "CRRA": 2.0,
    "DiscFac": 0.975,
    # State-dependent prices and survival (quarterly)
    "Rfree": [np.array([1.04**0.25, 1.01**0.25])],  # ~4% vs ~1% annual
    "LivPrb": [np.array([0.99375, 0.99375])],  # ~2.5% annual mortality
    "PermGroFac": [np.array([1.0, 1.0])],  # no growth => 1D grid suffices
    # Income process (same in both states except UnempPrb)
    "PermShkStd": np.array([[0.06, 0.06]]),
    "PermShkCount": 5,
    "TranShkStd": np.array([[0.2, 0.2]]),
    "TranShkCount": 5,
    "UnempPrb": np.array([0.02, 0.12]),  # 2% in expansion, 12% in contraction
    "IncUnemp": np.array([0.3, 0.3]),
    "T_retire": 0,
    "UnempPrbRet": None,
    "IncUnempRet": None,
    # Markov structure: symmetric persistence
    "Mrkv_p11": [p_stay],
    "Mrkv_p22": [p_stay],
    "MrkvPrbsInit": np.array([0.5, 0.5]),
    "global_markov": False,
    # Borrowing constraint and asset grid
    "BoroCnstArt": 0.0,
    "aXtraMin": 0.001,
    "aXtraMax": 20,
    "aXtraNestFac": 3,
    "aXtraCount": 48,
    "aXtraExtra": None,
    # MC simulation parameters
    "AgentCount": 100_000,
    "T_sim": 1100,
    "kLogInitMean": -12.0,
    "kLogInitStd": 0.0,
    "kNrmInitCount": 15,
    "pLogInitMean": 0.0,
    "pLogInitStd": 0.0,
    "pLvlInitCount": 15,
    "PermGroFacAgg": 1.0,
    "NewbornTransShk": False,
    "PerfMITShk": False,
    "neutral_measure": False,
    "T_cycle": 1,
    "cycles": 0,
}

print("Parameters set.")

## 2. Solve the Model

In [ ]:
agent = MarkovConsumerType(**params)
agent.solve()

MrkvArr = agent.MrkvArray[0]
J = MrkvArr.shape[0]

print(f"Number of Markov states: {J}")
print(f"Solution has {len(agent.solution[0].cFunc)} consumption functions")
print("\nMrkvArray (row-stochastic):")
print(MrkvArr)
print(f"Row sums: {MrkvArr.sum(axis=1)}")

# Stationary distribution of the Markov chain
eigvals, eigvecs = np.linalg.eig(MrkvArr.T)
idx = np.argmin(np.abs(eigvals - 1.0))
markov_stationary = eigvecs[:, idx].real
markov_stationary = markov_stationary / markov_stationary.sum()
print(f"Stationary Markov distribution: {markov_stationary}")

In [ ]:
# [consumption_functions_by_state]
m_plot = np.linspace(0.01, 10, 200)

plt.figure(figsize=(10, 6))
for j in range(J):
    c_vals = agent.solution[0].cFunc[j](m_plot)
    label = ["Expansion (j=0)", "Contraction (j=1)"][j]
    plt.plot(m_plot, c_vals, label=label, linewidth=2)
plt.plot(m_plot, m_plot, "--", color="gray", alpha=0.5, label="45-degree line")
plt.xlabel("Market resources $m$")
plt.ylabel("Consumption $c$")
plt.title("Consumption Functions by Markov State")
plt.legend()
plt.xlim([0, 10])
plt.ylim([0, 6])
plt.tight_layout()
plt.show()

## 3. Monte Carlo Simulation

Run the built-in MC simulator and compute aggregate consumption and assets.

In [ ]:
agent.track_vars = ["aNrm", "cNrm", "mNrm", "pLvl", "Mrkv"]
agent.initialize_sim()
# Avoid newborn transitory-shock suppression: HARK forces TranShk=1.0 for
# agents with t_age=0 (when NewbornTransShk=False), which would bias period 0.
agent.t_age = np.ones(agent.AgentCount, dtype=int)

t0_mc = time.time()
agent.simulate()
mc_sim_time = time.time() - t0_mc

MC_C = np.mean(agent.state_now["mNrm"] - agent.state_now["aNrm"])
MC_A = np.mean(agent.state_now["aNrm"])

print(
    f"MC simulation: {mc_sim_time:.2f}s ({agent.AgentCount:,} agents, {agent.T_sim} periods)"
)
print(f"MC Aggregate Consumption = {MC_C:.6f}")
print(f"MC Aggregate Assets      = {MC_A:.6f}")

for j in range(J):
    frac = np.mean(agent.shocks["Mrkv"] == j)
    print(f"Fraction in state {j}: {frac:.3f} (expected {markov_stationary[j]:.3f})")

In [ ]:
mc_aLvls = np.array([np.mean(agent.history["aNrm"][t]) for t in range(agent.T_sim)])

## 4. Transition Matrix Construction (Ad-Hoc Prototype)

We build a transition matrix over the joint state $(m, j)$ where $m$ is normalized
market resources on a grid of $M$ points and $j \in \{0, 1\}$ is the Markov state.
The full state vector has $M \times J$ entries.

**Key convention:** The Markov transition is *row-stochastic*, so
`MrkvArr[j, jp]` = P(transition from state $j$ to state $j'$).

In [ ]:
mMin = 0.001
mMax = 50
mCount = 200
mFac = 3

dist_mGrid = make_grid_exp_mult(ming=mMin, maxg=mMax, ng=mCount, timestonest=mFac)

M = len(dist_mGrid)
N_states = M * J

print(f"m-grid: {M} points from {dist_mGrid[0]:.4f} to {dist_mGrid[-1]:.1f}")
print(f"Markov states: {J}")
print(f"Total states in TM: {N_states}")

In [ ]:
start = time.time()

Rfree_arr = agent.Rfree[0]
LivPrb_arr = agent.LivPrb[0]
PermGroFac_arr = agent.PermGroFac[0]
IncShkDstn_list = agent.IncShkDstn[0]

# Policy on m-grid for each Markov state
cPol = []
aPol = []
for j in range(J):
    c_j = agent.solution[0].cFunc[j](dist_mGrid)
    a_j = np.maximum(dist_mGrid - c_j, 0.0)
    cPol.append(c_j)
    aPol.append(a_j)

# Newborn distribution over (m, j)
# Newborns: a=0 => m = TranShk; Markov state from stationary dist
MrkvPrbsInit = markov_stationary
NewBornDist = np.zeros(N_states)
for jp in range(J):
    shk_dstn_jp = IncShkDstn_list[jp]
    newborn_m = jump_to_grid_1D(shk_dstn_jp.atoms[1], shk_dstn_jp.pmv, dist_mGrid)
    NewBornDist[jp * M : (jp + 1) * M] = MrkvPrbsInit[jp] * newborn_m

# Build transition matrix
# Convention: MrkvArr[j, jp] = P(j -> jp)  [row-stochastic]
TranMatrix = np.zeros((N_states, N_states))

for j in range(J):
    a_grid = aPol[j]
    LivPrb_j = LivPrb_arr[j]

    for jp in range(J):
        markov_prob = MrkvArr[j, jp]  # P(j -> jp)
        if markov_prob < 1e-15:
            continue

        Rfree_jp = Rfree_arr[jp]
        PermGroFac_jp = PermGroFac_arr[jp]
        shk_dstn_jp = IncShkDstn_list[jp]
        shk_prbs = shk_dstn_jp.pmv
        perm_shks = shk_dstn_jp.atoms[0]
        tran_shks = shk_dstn_jp.atoms[1]
        bNext = Rfree_jp * a_grid

        for i in range(M):
            mNext = bNext[i] / (perm_shks * PermGroFac_jp) + tran_shks
            lottery_weights = jump_to_grid_1D(mNext, shk_prbs, dist_mGrid)
            src_idx = j * M + i
            TranMatrix[jp * M : (jp + 1) * M, src_idx] += (
                markov_prob * LivPrb_j * lottery_weights
            )

    for i in range(M):
        src_idx = j * M + i
        TranMatrix[:, src_idx] += (1.0 - LivPrb_j) * NewBornDist

tm_build_time = time.time() - start
print(f"Transition matrix built in {tm_build_time:.2f}s")
print(f"Shape: {TranMatrix.shape}")
col_sums = TranMatrix.sum(axis=0)
print(f"Column sums: min={col_sums.min():.8f}, max={col_sums.max():.8f}")

## 5. Ergodic Distribution

In [ ]:
start = time.time()

eigenvalues, eigenvectors = sp_linalg.eigs(
    TranMatrix, k=1, which="LM", v0=np.ones(N_states)
)
ergodic_dist = eigenvectors[:, 0].real
ergodic_dist = ergodic_dist / ergodic_dist.sum()

tm_ergo_time = time.time() - start
print(f"Ergodic distribution computed in {tm_ergo_time:.2f}s")

for j in range(J):
    mass_j = ergodic_dist[j * M : (j + 1) * M].sum()
    print(
        f"TM mass in state {j}: {mass_j:.4f} (Markov stationary: {markov_stationary[j]:.4f})"
    )

tm_total_time = tm_build_time + tm_ergo_time
print("\n--- Timing Summary ---")
print(f"MC simulation:    {mc_sim_time:.2f}s ({agent.AgentCount:,} agents)")
print(f"TM build + ergo:  {tm_total_time:.2f}s ({mCount} m-pts × {J} states)")
print(f"Speedup:          {mc_sim_time / tm_total_time:.1f}×")

In [ ]:
TM_C = 0.0
TM_A = 0.0
for j in range(J):
    p_j = ergodic_dist[j * M : (j + 1) * M]
    TM_C += np.dot(cPol[j], p_j)
    TM_A += np.dot(aPol[j], p_j)

print(f"TM Aggregate Consumption = {TM_C:.6f}")
print(f"TM Aggregate Assets      = {TM_A:.6f}")

## 6. Comparison

In [ ]:
print("=== Aggregate Comparison ===")
print(f"{'':20s} {'MC':>12s} {'TM':>12s} {'Diff':>12s}")
print(f"{'Consumption':20s} {MC_C:12.6f} {TM_C:12.6f} {MC_C - TM_C:12.6f}")
print(f"{'Assets':20s} {MC_A:12.6f} {TM_A:12.6f} {MC_A - TM_A:12.6f}")

### Time series comparison

MC aggregate assets fluctuate (sampling noise); TM gives a flat line at the ergodic mean.

In [ ]:
# [mc_vs_tm_asset_paths]
dstn = ergodic_dist.copy()
tm_aLvls = []
for t in range(agent.T_sim - BURNIN):
    A_val = sum(np.dot(aPol[j], dstn[j * M : (j + 1) * M]) for j in range(J))
    tm_aLvls.append(A_val)
    dstn = TranMatrix @ dstn

n_agents = agent.AgentCount
plt.figure(figsize=(16, 6))
plt.plot(
    mc_aLvls[BURNIN:],
    color=COLOR_MC,
    alpha=0.7,
    linewidth=0.8,
    label=f"MC ({n_agents:,} agents)",
)
plt.plot(
    tm_aLvls, color=COLOR_TM, linewidth=2.5, label=f"TM ({mCount} m-pts × {J} states)"
)
plt.xlabel("Period (after burn-in)")
plt.ylabel("Aggregate Assets (normalized)")
plt.title("MC vs TM: Aggregate Assets Time Series")
plt.legend(fontsize=12)
plt.tight_layout()
plt.show()

### Distribution of normalized market resources by Markov state

Compare the TM ergodic distribution with the MC histogram, separately for each state.

In [ ]:
# [dist_normalized_market_resources_by_state]
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

# Compute TM bin widths for converting probability mass to density
bin_widths = np.diff(dist_mGrid)
bin_centers = 0.5 * (dist_mGrid[:-1] + dist_mGrid[1:])

for j in range(J):
    ax = axes[j]
    p_j = ergodic_dist[j * M : (j + 1) * M]
    mass_j = p_j.sum()
    p_j_cond = p_j / mass_j if mass_j > 0 else p_j

    # Convert TM probability mass to density by dividing by bin widths.
    # Use midpoint-rule bins: mass at grid point i spans [g_{i-1/2}, g_{i+1/2}].
    midpoint_widths = np.zeros(M)
    midpoint_widths[0] = dist_mGrid[1] - dist_mGrid[0]
    midpoint_widths[-1] = dist_mGrid[-1] - dist_mGrid[-2]
    midpoint_widths[1:-1] = 0.5 * (dist_mGrid[2:] - dist_mGrid[:-2])
    tm_density = p_j_cond / midpoint_widths

    ax.plot(
        dist_mGrid,
        tm_density,
        color=COLOR_TM,
        linewidth=2,
        label=f"TM ({mCount} m-pts)",
    )

    # MC histogram with uniform bins, plotted as density
    in_state_j = agent.shocks["Mrkv"] == j
    mc_m_j = agent.state_now["mNrm"][in_state_j]
    ax.hist(
        mc_m_j,
        bins=N_MC_BINS,
        density=True,
        alpha=0.4,
        color=COLOR_MC,
        label=f"MC ({n_agents:,} agents)",
    )

    state_name = ["Expansion (j=0)", "Contraction (j=1)"][j]
    ax.set_title(f"{state_name}\n(mass = {mass_j:.3f})", fontsize=12)
    ax.set_xlabel("Normalized market resources $m$")
    ax.set_xlim([0, 15])
    ax.legend()

axes[0].set_ylabel("Probability Density")
plt.suptitle("Distribution of $m$ by Markov State", fontsize=14)
plt.tight_layout()
plt.show()

### Grid convergence

Show that TM aggregates converge toward the MC value as the grid becomes finer.

In [ ]:
grid_sizes = [50, 100, 200, 400]
tm_assets_by_grid = []

for mC in grid_sizes:
    g = make_grid_exp_mult(ming=mMin, maxg=mMax, ng=mC, timestonest=mFac)
    M_g = len(g)
    N_g = M_g * J

    cP = [agent.solution[0].cFunc[j](g) for j in range(J)]
    aP = [np.maximum(g - cP[j], 0.0) for j in range(J)]

    NBD = np.zeros(N_g)
    for jp in range(J):
        sd = IncShkDstn_list[jp]
        nb_m = jump_to_grid_1D(sd.atoms[1], sd.pmv, g)
        NBD[jp * M_g : (jp + 1) * M_g] = MrkvPrbsInit[jp] * nb_m

    TM_g = np.zeros((N_g, N_g))
    for j in range(J):
        a_g = aP[j]
        LivPrb_j = LivPrb_arr[j]
        for jp in range(J):
            mp = MrkvArr[j, jp]  # row-stochastic
            if mp < 1e-15:
                continue
            Rfp = Rfree_arr[jp]
            PGFp = PermGroFac_arr[jp]
            sd = IncShkDstn_list[jp]
            bN = Rfp * a_g
            for i in range(M_g):
                mN = bN[i] / (sd.atoms[0] * PGFp) + sd.atoms[1]
                lw = jump_to_grid_1D(mN, sd.pmv, g)
                TM_g[jp * M_g : (jp + 1) * M_g, j * M_g + i] += mp * LivPrb_j * lw
        for i in range(M_g):
            TM_g[:, j * M_g + i] += (1.0 - LivPrb_j) * NBD

    ev, evec = sp_linalg.eigs(TM_g, k=1, which="LM", v0=np.ones(N_g))
    ed = evec[:, 0].real
    ed = ed / ed.sum()

    A_tm = sum(np.dot(aP[j], ed[j * M_g : (j + 1) * M_g]) for j in range(J))
    tm_assets_by_grid.append(A_tm)
    print(f"mCount={mC:4d}  TM Assets = {A_tm:.6f}")

print(f"MC Assets = {MC_A:.6f}")

In [ ]:
# [grid_convergence_assets]
import matplotlib.cm as cm

orange_shades = cm.Oranges(np.linspace(0.3, 0.9, len(grid_sizes)))

plt.figure(figsize=(10, 6))
for idx, mC in enumerate(grid_sizes):
    plt.axhline(
        y=tm_assets_by_grid[idx],
        linestyle="--",
        alpha=0.8,
        color=orange_shades[idx],
        label=f"TM ({mC} m-pts)",
    )
plt.axhline(y=MC_A, color=COLOR_MC, linewidth=2, label=f"MC mean ({n_agents:,} agents)")
plt.ylabel("Aggregate Assets")
plt.title("Grid Convergence: TM Assets vs Grid Resolution")
plt.legend()
plt.tight_layout()
plt.show()

## 7. Summary

This prototype demonstrates that transition matrix methods extend naturally to
`MarkovConsumerType`.  The key differences from the non-Markov case (Du's notebook):

1. **Joint state space:** The distribution is tracked over $(m, j)$ — market resources
   $\times$ Markov state — giving a transition matrix of size $(M \times J)^2$.

2. **State-dependent transitions:** Each column of $\boldsymbol{\Pi}$ sums over all
   possible Markov transitions $j \to j'$, weighted by $\pi_{jj'}$.  The income shocks
   and interest rate depend on the *target* state $j'$.

3. **State-dependent policy:** The consumption function $c_j^*(m)$ depends on the
   current Markov state, so the asset grid $a = m - c_j^*(m)$ differs by state.

4. **Branching in the Markov dimension:** The Markov transition is a discrete branching
   event — no lottery is needed in the $j$ dimension (it's already discrete).  The lottery
   is only applied in the $m$ dimension.

### Next steps

- Refactor this ad-hoc code into a `calc_transition_matrix` method on `MarkovConsumerType`
- Add Harmenberg's neutral measure support
- Extend to finite-horizon / MIT shock experiments
- Test with more Markov states ($J > 2$)